[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/adairgranados/adair-granados?quickstart=1)


<!-- gen:values.yml#weeks.08.notebook-title|Introduction -->
# Week 8: DataViz II: Storytelling, decluttering, and accessibility
<!-- /gen -->

*Editing a raw chart until it's ready to present to a client.*

## This week

Your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate)'s default charts can be cluttered, hard for some readers to read, and titled with the variable names. This notebook teaches the edits that turn such a draft into a figure a client can use. Remember to consult the [Runnymede Standard](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), the module's design guide.

### Example

Let's begin with an example. The dataset below belongs to **Ferry Lane Stores**, a five-branch convenience chain. It contains two years of monthly revenue (in-store and online), monthly footfall, and a mystery-shopper score per branch. I built the data so that total revenue falls for a year. A **click-and-collect relaunch in October 2025** reverses the fall. Online growth then covers what in-store keeps losing.

Run the cell below first.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(5813)  # module seed: Do not change

months = pd.date_range("2024-07-01", periods=24, freq="MS")
t = np.arange(24)

# Ferry Lane Stores: Monthly revenue by channel (£k) and footfall (000s of visits).
instore = 430 - 5.0 * t + rng.normal(0, 6, 24)
online = 90 + 2.0 * t + np.where(t >= 15, 8.0 * (t - 15), 0) + rng.normal(0, 4, 24)
footfall = 210 - 1.6 * t + rng.normal(0, 3, 24)

shop = pd.DataFrame(
    {"instore": instore.round(1), "online": online.round(1),
     "footfall": footfall.round(1)},
    index=months,
)
shop["revenue"] = (shop["instore"] + shop["online"]).round(1)

# Mystery-shopper scores, latest wave (0-100)
scores = pd.Series(
    {"Egham": 94.6, "Staines": 92.1, "Windsor": 91.4, "Woking": 90.8, "Camberley": 90.2},
    name="score",
)

RELAUNCH = pd.Timestamp("2025-10-01")
print(f"Created shop ({len(shop)} months x {shop.shape[1]} columns) and scores ({len(scores)} branches), in-memory, seed 5813")

## 1. Takeaway titles

Most readers read the title, and many read ... well ... not much else. Consultants use a **takeaway title** that clearly states the finding. "Monthly revenue, 2024-2026" describes the *variables*. "Click-and-collect has stopped the revenue slide" states the *finding*. The [design guide, section 5](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), shows the test for a takeaway title. Recommendation: If you cannot state the finding in one sentence, go back to the analysis before you design the figure.

The cell below draws the same figure twice, with a different title on each. [`ax.set_title`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_title.html) with `loc="left"` left-aligns each title. [`fig.autofmt_xdate()`](https://matplotlib.org/stable/api/figure_api.html#matplotlib.figure.Figure.autofmt_xdate) tilts the date labels so they do not overlap.

In [ ]:
#| fig-alt: "Two identical line charts of a shop's monthly revenue in thousands of pounds from July 2024 to June 2026, falling through 2025 and recovering from late 2025. The left chart is titled with a description, monthly revenue, July 2024 to June 2026; the right chart is titled with the takeaway, click-and-collect has stopped the revenue slide."
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax in axes:
    ax.plot(shop.index, shop["revenue"], color="#0072B2", linewidth=2.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel("Monthly revenue (£k)")

axes[0].set_title("Monthly revenue, Jul 2024–Jun 2026", loc="left")
axes[1].set_title("Click-and-collect has stopped the revenue slide",
                  loc="left", fontweight="bold")
fig.autofmt_xdate(rotation=30, ha="right")
plt.tight_layout()
plt.show()

But remember: A takeaway title **is a claim**, so it can overreach. "Click-and-collect will double revenue by 2027" claims more than this data shows. Keep the title to what the figure shows.

## 2. The declutter sequence

The [design guide, section 3](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), sets out the declutter sequence. Make its edits in order on every draft. Give each step to your Associate as a short instruction, then check the result. The [Demonstration](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/08/Demonstration.ipynb) applies the steps to one figure, one at a time.

The cell below shows an Associate's default draft on the left, and the same data after the whole sequence on the right. The direct labels use [`ax.annotate`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.annotate.html) with `textcoords="offset points"`, which places each label a few points to the right of the line's end.

In [ ]:
# Before and after: on the left, a cluttered line chart of in-store and online revenue with a grey background,
# a legend and a generic title; on the right, the same two lines with the online series in blue and the in-store series in grey,
# each labelled directly at its end, no legend and no background.

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

ax = axes[0]
ax.plot(shop.index, shop["instore"], linewidth=1.8, label="instore")
ax.plot(shop.index, shop["online"], linewidth=1.8, label="online")
ax.set_facecolor("#eaeaf2")
ax.grid(True, color="white", linewidth=1.0)
ax.legend(loc="center right")
ax.set_ylabel("value")
ax.set_title("Line Plot of Data by Month")

ax = axes[1]
ax.plot(shop.index, shop["online"], color="#0072B2", linewidth=2.4)
ax.plot(shop.index, shop["instore"], color="#bdbdbd", linewidth=1.4)
ax.annotate("Online", (shop.index[-1], shop["online"].iloc[-1]),
            xytext=(8, 0), textcoords="offset points", va="center",
            color="#0072B2", fontsize=9, fontweight="bold")
ax.annotate("In-store", (shop.index[-1], shop["instore"].iloc[-1]),
            xytext=(8, 0), textcoords="offset points", va="center",
            color="#767676", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.margins(x=0.02)
ax.set_ylabel("Monthly revenue (£k)")
ax.set_title("Online is closing the gap on in-store", loc="left", fontweight="bold")

fig.autofmt_xdate(rotation=30, ha="right")
plt.tight_layout()
plt.show()

Each change on the right **removes**, **relabels**, or **recolours** part of the chart. In [TRACE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-trace), these edits come under R, for **Reduce**.

⚠️ Step 4 of the sequence, **replace, never delete**, swaps the legend for direct labels. Step 6 turns the context lines grey and keeps them on the figure. If your Associate removes a legend without adding labels, or removes error bars because they look untidy, it has deleted information.

## 3. Annotation as narration

Annotation puts **your words** on the figure itself. This is what makes a 'default' chart a 'customised' one the client recognises as **made for them**. The [design guide, section 5](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), explains the usual kinds: Mark the event, name the difference, and say the "so what".

The cell below puts two of these kinds on a chart of Ferry Lane's total revenue. [`ax.axvline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axvline.html) draws a dotted line at the relaunch, `ax.annotate` adds a label with an arrow, and [`np.polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html) computes the "so what" from the data. `ax.text` then places that number in the corner with `transform=ax.transAxes`. That setting measures the position in axes coordinates, from 0 to 1 across the plot.

In [ ]:
#| fig-alt: "A line chart of total monthly revenue in thousands of pounds, falling through 2025 and recovering after a dotted vertical line marking the click-and-collect relaunch in October 2025, with an annotation naming the relaunch and a note giving the recovery rate per month since it."
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(shop.index, shop["revenue"], color="#0072B2", linewidth=2.2)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylabel("Total monthly revenue (£k)")

# Mark the event
ax.axvline(RELAUNCH, color="#767676", linewidth=0.8, linestyle=":")
ax.annotate("Click-and-collect relaunch\n(Oct 2025)",
            xy=(RELAUNCH, shop.loc[RELAUNCH, "revenue"]),
            xytext=(-95, 45), textcoords="offset points",
            fontsize=9, color="#444444",
            arrowprops=dict(arrowstyle="->", color="#767676"))

# The slope of a straight line fitted since the relaunch, in £k a month
post = shop.loc[RELAUNCH:, "revenue"]
recovery = np.polyfit(np.arange(len(post)), post, 1)[0]
ax.text(0.98, 0.08, f"recovering ~£{recovery:.0f}k/month since",
        transform=ax.transAxes, ha="right", fontsize=9, color="#444444")

ax.set_title("Click-and-collect has stopped the revenue slide",
             loc="left", fontweight="bold")
plt.tight_layout()
plt.show()

*(This also shows a challenge with annotations: As the data changes, the position of the label sometimes overlaps with the chart, which doesn't look right. You may have to experiment a bit here.)*

## 4. Colour: Semantics and colour-blindness

Business readers take a meaning from every colour. They read red as loss, green as gain, grey as background, and the brightest colour as what matters. Make these <!-- plain-language: everyday decision -->decisions on every figure.

First, decide what kind of palette the data needs.

| Palette type | Use for | Example |
|---|---|---|
| **Sequential** | Ordered values, low to high | Revenue by region, light to dark blue |
| **Diverging** | Values around a meaningful midpoint | Change against budget, red through white to blue |
| **Qualitative** | Unordered categories | Product lines, distinct hues with no implied order |

A rainbow palette on categories suggests they have an order. A qualitative palette on ordered values hides their order. [ColorBrewer](https://colorbrewer2.org) sorts its palettes this way and has a colour-blind-safe filter. <!-- docs-lint: allow: ColorBrewer is a proper noun -->

Second, check that everyone in the room can read it, under the rules in the [design guide, section 4](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md). For most business figures, use **grey for context and one Okabe-Ito accent for the story**. Do not rely on colour alone: Add direct labels, markers, line styles, or position.

The cell below draws the Okabe-Ito palette as a strip of [`Rectangle`](https://matplotlib.org/stable/api/_as_gen/matplotlib.patches.Rectangle.html) swatches.

In [ ]:
#| fig-alt: "A strip of eight colour swatches from the Okabe-Ito colour-blind-safe palette, each labelled with its name and hex code: black, orange, sky blue, bluish green, yellow, blue, vermillion, and reddish purple."
# The Okabe-Ito palette. Copy these hex codes into any figure you make this term.
okabe_ito = {
    "black": "#000000", "orange": "#E69F00", "sky blue": "#56B4E9",
    "bluish green": "#009E73", "yellow": "#F0E442", "blue": "#0072B2",
    "vermillion": "#D55E00", "reddish purple": "#CC79A7",
}

fig, ax = plt.subplots(figsize=(10, 1.9))
for i, (name, hexcode) in enumerate(okabe_ito.items()):
    ax.add_patch(plt.Rectangle((i, 0), 0.92, 1.0, color=hexcode))
    ax.text(i + 0.46, -0.22, name, ha="center", va="top", fontsize=9)
    ax.text(i + 0.46, 1.08, hexcode, ha="center", va="bottom",
            fontsize=8, family="monospace")
ax.set_xlim(-0.05, 8)
ax.set_ylim(-0.75, 1.55)
ax.axis("off")
ax.set_title("The Okabe-Ito colour-blind-safe palette", loc="left", pad=16)
plt.show()

## 5. Truth! axes

The [design guide, section 5](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), sets the rules for baselines and dual axes.

### Bars start at zero

**No exceptions.**

Bars encode value as **length**, so they start at zero. On a log scale, bars of ratios start at 1, as the guide explains. If you start the axis at 89, a 4.4-point difference looks like a huge gap. The cell below draws five mystery-shopper scores twice with [`Series.plot.bar`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.bar.html). The left chart cuts the axis to 89 with [`ax.set_ylim`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_ylim.html). The right chart starts from zero.

In [ ]:
# Two bar charts of the same five branches' mystery-shopper scores.
# On the left the vertical axis starts at 89, so one bar towers over the rest;
# on the right the axis starts at zero, and the five bars are within 4.4 points of each other.

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

scores.plot.bar(ax=axes[0], color="#0072B2")
axes[0].set_ylim(89, 95)   # ⚠️ Truncated axis: Do not use this in a report
axes[0].set_title("Cut at 89: A 4.4-point gap looks huge", loc="left")

scores.plot.bar(ax=axes[1], color="#0072B2")
axes[1].set_ylim(0, 100)
axes[1].set_title("From zero: Five branches within 4.4 points", loc="left")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylabel("Mystery-shopper score (0-100)")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

*(Did I say "No exceptions!"?)*

### Lines not so much

Lines encode value as **position and slope**, so a line chart may crop its axis. Cropping is often the only way to show the variation the reader needs. The test is <!-- plain-language: everyday intent --> intent: A cropped axis should show **variation that matters to the reader**. If the crop makes a small change look large, start from zero. If you crop, say so on the figure ("axis begins at £440k"), and explain the choice in your [design rationale](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-design-rationale).

The cell below draws Ferry Lane's revenue from zero and cropped to the data.

In [ ]:
# Two line charts of the same monthly revenue series. On the left the vertical axis starts at zero and the line looks flat;
# on the right the axis is cropped to the data, and the fall and the recovery are visible.

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(shop.index, shop["revenue"], color="#0072B2", linewidth=2)
axes[0].set_ylim(0, 560)
axes[0].set_title("From zero: The line looks flat", loc="left")

axes[1].plot(shop.index, shop["revenue"], color="#0072B2", linewidth=2)
axes[1].set_title("Cropped to the data: The fall and the recovery are visible", loc="left")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel("Total monthly revenue (£k)")
fig.autofmt_xdate(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Dual y-axes: Two free scales

Two independent y-axes can make two series look as if they rise together. If you change the scales, the same two series can look as if one rises while the other falls. The [design guide, section 5](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), gives better options. Index both series to 100 at a common start, or use two stacked panels that share an x-axis.

The cell below shows revenue and footfall both ways. The left panel uses [`ax.twinx()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.twinx.html), which adds a second y-axis on the right. The right panel indexes both series to 100, and [`ax.axhline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axhline.html) draws the line at 100.

In [ ]:
# Two charts of revenue and footfall over the same months.
# On the left, a dual-axis chart with revenue in blue on the left axis and footfall
# in orange on the right axis, where the two scales were chosen by the chart maker.
# On the right, both series indexed to 100 at the first month on one shared axis,
# labelled directly at their ends, with a horizontal line at 100.

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Left: The dual-axis version your Associate offers by default
ax = axes[0]
ax.plot(shop.index, shop["revenue"], color="#0072B2", linewidth=2, label="Revenue (£k)")
ax.set_ylabel("Revenue (£k)", color="#0072B2")
ax2 = ax.twinx()
ax2.plot(shop.index, shop["footfall"], color="#D55E00", linewidth=2, label="Footfall (000s)")
ax2.set_ylabel("Footfall (000s)", color="#D55E00")
ax.set_title("Dual axes: How the lines relate depends on\ntwo scales the chart maker chose", loc="left")

# Right: Both series indexed to 100 at July 2024, on one shared scale
ax = axes[1]
for col, colour, label in [("revenue", "#0072B2", "Revenue"),
                           ("footfall", "#D55E00", "Footfall")]:
    indexed = 100 * shop[col] / shop[col].iloc[0]
    ax.plot(shop.index, indexed, color=colour, linewidth=2)
    ax.annotate(label, (shop.index[-1], indexed.iloc[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", color=colour, fontsize=9)
ax.axhline(100, color="#bbbbbb", linewidth=0.8, linestyle=":")
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.02)
ax.set_ylabel("Index (Jul 2024 = 100)")
ax.set_title("Indexed to 100: Footfall down ~18%,\nrevenue held, so spend per visit is up", loc="left")

plt.tight_layout()
plt.show()

The indexed panel puts both series on one scale, and it shows the finding: Fewer visitors are spending more per visit.

### Showing how sure you are

This week's [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) is about **uncertainty**. Some lines show an average of noisy data, such as mean takings per quarter or mean score per month. A good figure often carries a shaded **95% confidence band** around such a line. The band is roughly the range where the true average could sit. Where two bands overlap, the true difference between the lines may be zero.

Deleting bands or error bars to make a chart look cleaner is the most serious mistake on the list in the [design guide, section 5](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md). Keep the bands. Check whether the gap between the lines is bigger than the bands. Ask your Associate: *"What is a 95% confidence band, in one sentence a board member would accept?"* Keep that sentence: The Quest's [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver) and [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) stages use it.

## 6. Alt text: Try to express the figure in one sentence

Every figure you hand in or present carries **alt text**: The sentence a screen reader reads in place of the figure. The formula, from the [design guide, section 4](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md):

> **chart type + variables + the takeaway**, in that order, one sentence.

- ❌ *"A chart of revenue."* This gives no finding
- ❌ *"Line chart with a blue line, an x-axis showing months and a y-axis showing values."* This describes the pixels and still gives no finding
- ✅ *"Line chart of Ferry Lane Stores' total monthly revenue, July 2024 to June 2026, showing a year-long decline reversed after the October 2025 click-and-collect relaunch"*

Write the alt text right after the takeaway title. Keep it with the figure: A caption line in your report, or a comment or variable in your notebook. Each key figure also needs a longer description, which the same section of the guide explains.

## Where this goes next

This week's Quest is **[Quest 08](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/08/Quest.ipynb): The Boardroom Story**.

Keep the [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md) open all week: Sections 3 to 6 contain the rules this notebook demonstrates. Run TRACE over every key figure. Record in your [Prompt Diary](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-prompt-diary) how you directed each one from draft to decision-ready.